In [16]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer

In [17]:
df = pd.read_csv("diabetes_raw.csv")

In [18]:
zero_invalid_cols = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]
df[zero_invalid_cols] = df[zero_invalid_cols].replace(0, np.nan)

print("Missing values after marking disguised zeros as NaN:")
print(df.isnull().sum())

Missing values after marking disguised zeros as NaN:
Pregnancies                   0
Glucose                       5
BloodPressure                35
SkinThickness               227
Insulin                     374
BMI                          11
DiabetesPedigreeFunction      0
Age                           0
Outcome                       0
dtype: int64


In [19]:
# duplicates

n_dupes = df.duplicated().sum()
print(f"Duplicate rows: {n_dupes}")

Duplicate rows: 0


In [20]:
# split features/target

X = df.drop(columns=["Outcome"])
y = df["Outcome"]

In [21]:
# train/test split (stratified to preserve 65/35 class balance)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print(f"\nTrain shape: {X_train.shape}, Test shape: {X_test.shape}")

# median imputation - fit on TRAIN ONLY to avoid data leakage
imputer = SimpleImputer(strategy="median")
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X.columns, index=X_train.index)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X.columns, index=X_test.index)

# feature scaling - fit on TRAIN ONLY
scaler = StandardScaler()
X_train_scaled = pd.DataFrame(scaler.fit_transform(X_train_imp), columns=X.columns, index=X_train.index)
X_test_scaled = pd.DataFrame(scaler.transform(X_test_imp), columns=X.columns, index=X_test.index)

# Save everything needed for modelling stage
X_train_imp.to_csv("X_train_imputed.csv", index=False)
X_test_imp.to_csv("X_test_imputed.csv", index=False)
X_train_scaled.to_csv("X_train_scaled.csv", index=False)
X_test_scaled.to_csv("X_test_scaled.csv", index=False)
y_train.to_csv("y_train.csv", index=False)
y_test.to_csv("y_test.csv", index=False)

print("\nPreprocessing complete.")
print("Train class balance:\n", y_train.value_counts(normalize=True))
print("Test class balance:\n", y_test.value_counts(normalize=True))



Train shape: (614, 8), Test shape: (154, 8)

Preprocessing complete.
Train class balance:
 Outcome
0    0.651466
1    0.348534
Name: proportion, dtype: float64
Test class balance:
 Outcome
0    0.649351
1    0.350649
Name: proportion, dtype: float64
